In [ ]:
import numpy as np
import yfinance as yf
import pandas as pd

# List of selected stocks and ETFs
tickers = ["SOXL"]

# User input: Choose 'call', 'put', or 'both'
option_type = 'call'

# Fetch the first available expiration date for each stock
expiration_dates = {}
stock_prices = {}
stock_iv = {}

for ticker in tickers:
    stock = yf.Ticker(ticker)

    # Ensure the ticker has options available
    if stock.options:
        expiration_dates[ticker] = stock.options[0]  # Select the first available expiration
        stock_prices[ticker] = stock.history(period="1d")["Close"].iloc[-1]  # Get latest stock price

        # Compute X-day historical volatility as a proxy for implied volatility
        hist_data = stock.history(period="1mo")["Close"]
        log_returns = np.log(hist_data / hist_data.shift(1))
        hist_volatility = np.sqrt(252) * log_returns.std()  # Annualized volatility
        stock_iv[ticker] = hist_volatility

# Fetch and store option chains
options_data = []

for ticker in tickers:
    if ticker not in expiration_dates:  # Skip tickers without options
        print(f"Skipping {ticker} (No options available).")
        continue

    stock = yf.Ticker(ticker)
    expiration = expiration_dates[ticker]
    opt_chain = stock.option_chain(expiration)

    # Extract call and put options based on user input
    calls = opt_chain.calls.copy()
    puts = opt_chain.puts.copy()

    calls["Type"] = "Call"
    puts["Type"] = "Put"

    if option_type == "call":
        all_options = calls
    elif option_type == "put":
        all_options = puts
    else:
        all_options = pd.concat([calls, puts])

    all_options["Ticker"] = ticker
    all_options["Expiration"] = expiration
    all_options["Stock Price"] = stock_prices[ticker]

    # Calculate the ratio: Bid Price / |Strike Price - Stock Price|
    # Key components
    # bid/strike = returns
    # |strike-spot|/IV = risk
    all_options["reward"] = all_options["bid"] / all_options["strike"] * 100
    all_options["risk"] = abs(all_options["strike"] - stock_prices[ticker]) / all_options["impliedVolatility"]
    # market bull sell put formula: reward/(risk)^(X/risk)
    all_options["Bid |Strike - Stock| ratio"] = all_options["bid"] / all_options["strike"] * (abs(all_options["strike"] - stock_prices[ticker]) / stock_iv[ticker])**(stock_iv[ticker]/abs(all_options["strike"] - stock_prices[ticker])) * 100
    # all_options["Bid |Strike - Stock| ratio"] = all_options["bid"] / all_options["strike"] * (abs(all_options["strike"] - stock_prices[ticker]) / all_options["impliedVolatility"])**np.sqrt((abs(all_options["strike"] - stock_prices[ticker]) / all_options["impliedVolatility"])) * X

    # Filter for options with:
    # X. Reasonable bid price (> X)
    # X. Open interest > X (to ensure liquidity)
    # X. Strike price between X% and X% of stock price
    filtered_options = all_options[
        (all_options["bid"] > 0) &
        (all_options["openInterest"] > 0) &
        (all_options["strike"] >= 0.5 * stock_prices[ticker]) &
        (all_options["strike"] <= 1.5 * stock_prices[ticker])
        ]

    # Append to list
    options_data.append(filtered_options)

# Combine all data
if options_data:
    final_options = pd.concat(options_data)

    # Select relevant columns
    final_options = final_options[[
        "Ticker", "Stock Price", "Expiration", "Type", "strike", "bid",
        "impliedVolatility", "reward", "risk", "Bid |Strike - Stock| ratio", "volume", "openInterest"
    ]]

    # Categorize into two groups:
    # - Options with strike price above stock price
    # - Options with strike price below stock price
    above_stock_price = final_options[final_options["strike"] > final_options["Stock Price"]]
    below_stock_price = final_options[final_options["strike"] < final_options["Stock Price"]]

    # Sort by the new ratio (Highest to Lowest)
    above_stock_price = above_stock_price.sort_values(by=["Bid |Strike - Stock| ratio"], ascending=False)
    below_stock_price = below_stock_price.sort_values(by=["Bid |Strike - Stock| ratio"], ascending=False)

    # Save to separate CSV files
    above_stock_price.to_csv("options_above_stock_price.csv", index=False)
    below_stock_price.to_csv("options_below_stock_price.csv", index=False)

    # Display full columns
    pd.set_option("display.max_columns", None)  # Show all columns

    # Print results for verification
    print("Options with Strike Price Above Stock Price:")
    print(above_stock_price.to_string())  # Show everything without truncation

    print("\nOptions with Strike Price Below Stock Price:")
    print(below_stock_price.to_string())  # Show everything without truncation

else:
    print("No options data found for the given tickers.")


In [ ]:
import numpy as np
import yfinance as yf
import pandas as pd

# List of selected stocks and ETFs
tickers = ["SOXL"]

# Fetch the first available expiration date for each stock
expiration_dates = {}
stock_prices = {}
stock_iv = {}

for ticker in tickers:
    stock = yf.Ticker(ticker)

    # Ensure the ticker has options available
    if stock.options:
        expiration_dates[ticker] = stock.options[0]  # Select the first available expiration
        stock_prices[ticker] = stock.history(period="1d")["Close"].iloc[-1]  # Get latest stock price

        # Compute X-day historical volatility as a proxy for implied volatility
        hist_data = stock.history(period="1mo")["Close"]
        log_returns = np.log(hist_data / hist_data.shift(1))
        hist_volatility = np.sqrt(252) * log_returns.std()  # Annualized volatility
        stock_iv[ticker] = hist_volatility

# Fetch and store option chains
options_data = []

for ticker in tickers:
    if ticker not in expiration_dates:  # Skip tickers without options
        print(f"Skipping {ticker} (No options available).")
        continue

    stock = yf.Ticker(ticker)
    expiration = expiration_dates[ticker]
    opt_chain = stock.option_chain(expiration)

    # Extract call and put options based on user input
    calls = opt_chain.calls.copy()
    puts = opt_chain.puts.copy()

    calls["Type"] = "Call"
    puts["Type"] = "Put"

    filtered_calls = calls[
        (calls["bid"] > 0) &
        (calls["ask"] > 0) &
        (calls["openInterest"] > 0) &
        (calls["strike"] >= 0.5 * stock_prices[ticker]) &
        (calls["strike"] <= 1.5 * stock_prices[ticker])
    ]

    filtered_puts = puts[
        (puts["bid"] > 0) &
        (puts["ask"] > 0) &
        (puts["openInterest"] > 0) &
        (puts["strike"] >= 0.5 * stock_prices[ticker]) &
        (puts["strike"] <= 1.5 * stock_prices[ticker])
        ]

    # Append to list
    options_data.append(filtered_options)

# Combine all data
if options_data:
    final_options = pd.concat(options_data)

    # Select relevant columns
    final_options = final_options[[
        "Ticker", "Stock Price", "Expiration", "Type", "strike", "bid",
        "impliedVolatility", "reward", "risk", "Bid |Strike - Stock| ratio", "volume", "openInterest"
    ]]

    # Categorize into two groups:
    # - Options with strike price above stock price
    # - Options with strike price below stock price
    above_stock_price = final_options[final_options["strike"] > final_options["Stock Price"]]
    below_stock_price = final_options[final_options["strike"] < final_options["Stock Price"]]

    # Sort by the new ratio (Highest to Lowest)
    above_stock_price = above_stock_price.sort_values(by=["Bid |Strike - Stock| ratio"], ascending=False)
    below_stock_price = below_stock_price.sort_values(by=["Bid |Strike - Stock| ratio"], ascending=False)

    # Display full columns
    pd.set_option("display.max_columns", None)  # Show all columns

    # Print results for verification
    print("Options with Strike Price Above Stock Price:")
    print(above_stock_price.to_string())  # Show everything without truncation

    print("\nOptions with Strike Price Below Stock Price:")
    print(below_stock_price.to_string())  # Show everything without truncation

else:
    print("No options data found for the given tickers.")
